# Calcula tu inflación — por marca

Calcula la **inflación real por marca** con los datos abiertos de *Quién es Quién
en los Precios* (Profeco), y detecta **reduflación**: cuando el producto cuesta
casi lo mismo pero trae menos contenido.

**No necesitas saber Python ni instalar nada.** Ejecuta las celdas en orden con
el botón ▶ (o `Shift + Enter`).

### Antes de empezar

Sube tus `.rar` de QQP **sin descomprimir** a una carpeta `QQP` en tu Google Drive:

```
Mi unidad/
└── QQP/
    ├── QQP_2024.rar
    └── QQP_2025.rar
```

No los descomprimas: **un año de QQP ocupa cerca de 5 GB** y no cabría en el disco
de Colab. Este cuaderno saca una pieza a la vez, se queda solo con lo que pediste
y la borra antes de seguir — así nunca ocupa más de unos 100 MB.

---

## Paso 1 — Preparar

In [ ]:
#@title Paso 1: preparar
!pip install -q pandas
!apt-get -qq install -y unar > /dev/null 2>&1
!wget -q -O inflacion_por_marca.py https://raw.githubusercontent.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N/main/inflacion_por_marca.py
!wget -q -O colab_qqp.py https://raw.githubusercontent.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N/main/colab_qqp.py

import importlib, colab_qqp
importlib.reload(colab_qqp)
colab_qqp.preparar()

---

## Paso 2 — Conectar tu Google Drive

Google te va a pedir permiso: elige tu cuenta y acepta.

In [ ]:
#@title Paso 2: conectar Drive
from google.colab import drive
drive.mount("/content/drive")
print("\nDrive conectado.")

---

## Paso 3 — Ubicar la carpeta

Si los pusiste en `Mi unidad/QQP`, déjalo así. Te va a listar los `.rar` que
encontró — **anota los nombres exactos**, los necesitas en el Paso 4.

In [ ]:
#@title Paso 3: ubicar la carpeta
CARPETA_EN_DRIVE = "QQP"  #@param {type:"string"}

CARPETA = colab_qqp.ubicar(CARPETA_EN_DRIVE)

---

## Paso 3B — Asomarse a un archivo (opcional)

Si el Paso 5 falla diciendo que faltan columnas, ejecuta esto: saca **una sola
pieza** y te muestra cómo viene el archivo por dentro.

Es rápido, no descomprime el año completo.

In [ ]:
#@title Paso 3B: ver una pieza (opcional)
CUAL_COMPRIMIDO = "QQP_2024.rar"  #@param {type:"string"}

colab_qqp.ver_una_pieza(CARPETA, CUAL_COMPRIMIDO)

---

## Paso 4 — Elegir qué comparar

**Los dos `.rar`**: dos años distintos, tal como aparecieron en el Paso 3.

**El MES** (obligatorio): se aplica a los dos años. Comparar el mismo mes evita
confundir inflación con temporada. `8` = agosto.

**El filtro** (obligatorio): al menos un producto o categoría. Es lo que hace que
el proceso sea rápido y quepa en memoria.

> Tarda varios minutos: tiene que recorrer todas las piezas de ambos años.

In [ ]:
#@title Paso 4: qué comparar
ANIO_BASE   = "QQP_2024.rar"  #@param {type:"string"}
ANIO_ACTUAL = "QQP_2025.rar"  #@param {type:"string"}

MES = 8  #@param {type:"slider", min:0, max:12, step:1}

PRODUCTO  = "DESODORANTE"  #@param {type:"string"}
MARCA     = ""             #@param {type:"string"}
CATEGORIA = ""             #@param {type:"string"}
ESTADO    = ""             #@param {type:"string"}
CADENA    = ""             #@param {type:"string"}

POR_CADENA = True      #@param {type:"boolean"}
MIN_OBSERVACIONES = 3  #@param {type:"integer"}

FILTROS = dict(producto=PRODUCTO, marca=MARCA, categoria=CATEGORIA,
               estado=ESTADO, cadena=CADENA)
print("Listo. Pasa al Paso 5.")

---

## Paso 5 — Calcular

Verás el avance pieza por pieza.

In [ ]:
#@title Paso 5: calcular
colab_qqp.analizar(CARPETA, ANIO_BASE, ANIO_ACTUAL, MES, FILTROS,
                   por_cadena=POR_CADENA, min_obs=MIN_OBSERVACIONES)

---

## Paso 6 — Guardar el resultado

In [ ]:
#@title Paso 6: guardar y descargar
import os, shutil

if not os.path.exists("resultado.csv"):
    print("Todavia no hay resultado. Ejecuta el Paso 5.")
else:
    destino = os.path.join(CARPETA, "resultado.csv")
    shutil.copy("resultado.csv", destino)
    print(f"Guardado en tu Drive: {destino}")
    from google.colab import files
    files.download("resultado.csv")

---

## Cómo leer la tabla de reduflación

| Columna | Qué significa |
|---|---|
| **ETIQUETA** | Cuánto subió el precio que ves en el anaquel |
| **CONTENIDO** | Cuánto cambió el tamaño del empaque (negativo = encogió) |
| **REAL** | Cuánto subió el precio por gramo o mililitro |
| **BRECHA** | REAL menos ETIQUETA — la inflación que no se ve |

Una marca con `<-- ENCOGIO` redujo el contenido. Si su BRECHA es grande, estás
pagando bastante más por gramo aunque el precio casi no se haya movido.

## Notas

- Se usa la **mediana**, no el promedio, para que unos pocos registros mal
  capturados no distorsionen el resultado.
- Un artículo solo aparece si está en **ambos** periodos con al menos
  `MIN_OBSERVACIONES` registros.
- Tu Drive conserva los `.rar` y el `resultado.csv`; lo demás se borra al cerrar.

Código: <https://github.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N>